# WM-811K SSL 후보 train-OOF 정상 재현율 보정

앞선 자기지도 후보는 Macro-F1과 취약 클래스 recall을 개선했지만 none recall 안전 기준을 0.107%p 초과했습니다. 이 노트북은 보정값을 validation에서 고르지 않고 frozen train lot의 3-fold OOF 예측만으로 고정합니다.

- bias 선택: train lot OOF only
- OOF 모델: SSL encoder 초기화, fixed 10 epochs, 조기종료 없음
- validation: bias를 고정한 뒤 한 번만 평가
- test: 로드·평가·선택에 사용하지 않음
- 통과하더라도 자동 배포하지 않고 외부 holdout 후보로만 등록

In [ ]:
!pip install -q 'pandas==2.2.3' 'scikit-learn==1.6.1' Pillow

from google.colab import drive, files
from pathlib import Path
import json, shutil, subprocess, sys, zipfile
import torch

drive.mount('/content/drive')
assert torch.cuda.is_available(), 'T4 GPU 런타임을 선택하세요.'
print('GPU:', torch.cuda.get_device_name(0))

## 1. 실행 번들 하나 업로드

로컬 프로젝트의 노트북/wm811k_ssl_oof_calibration_bundle.zip을 선택합니다.

In [ ]:
uploaded = files.upload()
zip_names = [name for name in uploaded if name.lower().endswith('.zip')]
assert len(zip_names) == 1, f'ZIP 하나를 선택하세요: {list(uploaded)}'
bundle = Path('/content/wm811k_ssl_oof_bundle.zip')
bundle.write_bytes(uploaded[zip_names[0]])
bundle_dir = Path('/content/wm811k_ssl_oof_bundle')
if bundle_dir.exists(): shutil.rmtree(bundle_dir)
with zipfile.ZipFile(bundle) as archive:
    assert all('..' not in Path(name).parts for name in archive.namelist())
    archive.extractall(bundle_dir)
required = ['train_wm811k_cnn.py','calibrate_wm811k_robust_flat_bias.py','calibrate_wm811k_ssl_train_oof.py','split_assignments.csv','baseline_reproducibility_summary.json','baseline_class_recall_summary.json']
for name in required: assert (bundle_dir / name).is_file(), f'번들 누락: {name}'
print('번들 확인 완료')

## 2. 이전 자기지도 결과 확인

17번 노트북 결과가 Drive에 남아 있어야 합니다. 대용량 비라벨 배열은 이번 단계에서 다시 사용하지 않습니다.

In [ ]:
PROCESSED = Path('/content/drive/MyDrive/AI_project/WM811K_processed')
SSL_ROOT = Path('/content/drive/MyDrive/AI_project/WM811K_ssl_experiment/contrastive_v1')
for path in [PROCESSED/'wafer_maps_64.npy', PROCESSED/'labels.npy', SSL_ROOT/'ssl_encoder.pt']:
    assert path.is_file(), f'필요 파일이 없습니다: {path}'
for seed in [17,42,2026]:
    path = SSL_ROOT/'seeds'/f'seed_{seed}'/'validation_predictions.csv'
    assert path.is_file(), f'17번 노트북 결과가 없습니다: {path}'
print('이전 SSL 결과 확인 완료')

## 3. Train lot 3-fold OOF 보정 실행

T4 기준 약 15~30분 예상입니다. OOF none recall 99.5%를 만족하면서 취약 3종 recall 감소가 1%p 이내인 가장 작은 bias를 선택합니다.

In [ ]:
OUTPUT = Path('/content/drive/MyDrive/AI_project/WM811K_ssl_oof_calibration/train_oof_v1')
OUTPUT.mkdir(parents=True, exist_ok=True)
command = [sys.executable, str(bundle_dir/'calibrate_wm811k_ssl_train_oof.py'), '--maps', str(PROCESSED/'wafer_maps_64.npy'), '--labels', str(PROCESSED/'labels.npy'), '--assignments', str(bundle_dir/'split_assignments.csv'), '--ssl-encoder', str(SSL_ROOT/'ssl_encoder.pt'), '--validation-run-root', str(SSL_ROOT), '--baseline-summary', str(bundle_dir/'baseline_reproducibility_summary.json'), '--baseline-class-recall', str(bundle_dir/'baseline_class_recall_summary.json'), '--output-dir', str(OUTPUT), '--oof-folds', '3', '--epochs', '10', '--batch-size', '256', '--num-workers', '2', '--target-none-recall', '0.995', '--weak-recall-tolerance', '0.01']
subprocess.run(command, check=True, cwd=bundle_dir)
summary = json.loads((OUTPUT/'calibration_summary.json').read_text(encoding='utf-8'))
print('선택 bias:', summary['selected_none_logit_bias'])
print('결정:', summary['decision'])
print(json.dumps(summary, ensure_ascii=False, indent=2))

## 4. 전달용 결과 다운로드

행별 OOF/validation 예측과 checkpoint는 ZIP에 넣지 않습니다. 집계 지표와 bias sweep만 내려받습니다.

In [ ]:
zip_base = Path('/content/wm811k_ssl_oof_calibration_results')
zip_path = Path(shutil.make_archive(str(zip_base), 'zip', root_dir=OUTPUT))
print('ZIP:', zip_path, f'{zip_path.stat().st_size/1024:.1f} KB')
files.download(str(zip_path))